In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )
else:
    print("No GPU detected.")

PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

DATA_DIR = Path(
    "/content/drive/MyDrive/multi-agent-disinformation-detection/data/agent2"
)

print("Data directory exists:", DATA_DIR.exists())

for filename in [
    "train_scaled_sequences.npz",
    "validation_scaled_sequences.npz",
    "test_scaled_sequences.npz",
]:
    file_path = DATA_DIR / filename

    if file_path.exists():
        print(f"✓ {filename} — {file_path.stat().st_size / (1024**2):.2f} MB")
    else:
        print(f"✗ Missing: {filename}")

Data directory exists: True
✓ train_scaled_sequences.npz — 14.51 MB
✓ validation_scaled_sequences.npz — 3.06 MB
✓ test_scaled_sequences.npz — 3.07 MB


In [ ]:
!git clone --branch feature/agent2-data-pipeline \
https://github.com/kameldemri/multi-agent-disinformation-detection.git

Cloning into 'multi-agent-disinformation-detection'...
remote: Enumerating objects: 151, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 151 (delta 55), reused 121 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (151/151), 580.49 KiB | 11.16 MiB/s, done.
Resolving deltas: 100% (55/55), done.


In [ ]:
import sys

PROJECT_DIR = "/content/multi-agent-disinformation-detection"

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print("Project root added to Python path.")

Project root added to Python path.


In [ ]:
from agents.agent2_rnn.src.dataset import create_dataloader
from agents.agent2_rnn.src.model import AccountBiGRU
from agents.agent2_rnn.src.train import train_model, CONFIG

print("All imports successful!")
print("\nTraining configuration:")

for key, value in CONFIG.items():
    print(f"{key}: {value}")

All imports successful!

Training configuration:
input_size: 5
hidden_size: 64
num_layers: 1
dropout: 0.3
batch_size: 16
bucket_size_multiplier: 10
learning_rate: 0.001
weight_decay: 0.01
max_epochs: 30
patience: 5
gradient_clip: 1.0
threshold: 0.5
seed: 42


In [ ]:
from pathlib import Path
from agents.agent2_rnn.src import train

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/multi-agent-disinformation-detection/checkpoints/agent2"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Redirect the training controller's output directory
train.CHECKPOINT_DIR = OUTPUT_DIR

print("Checkpoint directory:", train.CHECKPOINT_DIR)
print("Directory exists:", train.CHECKPOINT_DIR.exists())

Checkpoint directory: /content/drive/MyDrive/multi-agent-disinformation-detection/checkpoints/agent2
Directory exists: True


In [ ]:
from pathlib import Path
from agents.agent2_rnn.src import preprocess

# Actual location of the processed datasets in Google Drive
DATA_DIR = Path(
    "/content/drive/MyDrive/multi-agent-disinformation-detection/data/agent2"
)

# Update the directory used by the dataset pipeline
preprocess.PROCESSED_DATA_DIR = DATA_DIR

print("Updated PROCESSED_DATA_DIR:", preprocess.PROCESSED_DATA_DIR)
print("Directory exists:", preprocess.PROCESSED_DATA_DIR.exists())

for split in ["train", "validation", "test"]:
    path = preprocess.PROCESSED_DATA_DIR / f"{split}_scaled_sequences.npz"
    print(f"{split}: {path.exists()} | {path}")

Updated PROCESSED_DATA_DIR: /content/drive/MyDrive/multi-agent-disinformation-detection/data/agent2
Directory exists: True
train: True | /content/drive/MyDrive/multi-agent-disinformation-detection/data/agent2/train_scaled_sequences.npz
validation: True | /content/drive/MyDrive/multi-agent-disinformation-detection/data/agent2/validation_scaled_sequences.npz
test: True | /content/drive/MyDrive/multi-agent-disinformation-detection/data/agent2/test_scaled_sequences.npz


In [ ]:
from agents.agent2_rnn.src.dataset import AccountSequenceDataset

train_dataset = AccountSequenceDataset("train")
validation_dataset = AccountSequenceDataset("validation")

Loading train dataset...
  Accounts: 1,451
  Total timesteps: 3,132,339
  Min length: 13
  Max length: 3,250
Loading validation dataset...
  Accounts: 311
  Total timesteps: 656,018
  Min length: 3
  Max length: 3,249


In [ ]:
print("TRAIN DATASET")
print("Number of accounts:", len(train_dataset))

print("\nVALIDATION DATASET")
print("Number of accounts:", len(validation_dataset))

TRAIN DATASET
Number of accounts: 1451

VALIDATION DATASET
Number of accounts: 311


In [ ]:
from agents.agent2_rnn.src.dataset import create_dataloader
from agents.agent2_rnn.src.train import CONFIG

train_loader = create_dataloader(
    split="train",
    batch_size=CONFIG["batch_size"],
    bucket_size_multiplier=CONFIG["bucket_size_multiplier"],
    seed=CONFIG["seed"],
)

validation_loader = create_dataloader(
    split="validation",
    batch_size=CONFIG["batch_size"],
    bucket_size_multiplier=CONFIG["bucket_size_multiplier"],
    seed=CONFIG["seed"],
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))

Loading train dataset...
  Accounts: 1,451
  Total timesteps: 3,132,339
  Min length: 13
  Max length: 3,250
Loading validation dataset...
  Accounts: 311
  Total timesteps: 656,018
  Min length: 3
  Max length: 3,249
Training batches: 91
Validation batches: 20


In [ ]:
# Retrieve one training batch
sequences, labels, lengths = next(iter(train_loader))

print("Sequences shape:", sequences.shape)
print("Labels shape:", labels.shape)
print("Lengths shape:", lengths.shape)

print("\nSequence data type:", sequences.dtype)
print("Labels data type:", labels.dtype)
print("Lengths data type:", lengths.dtype)

print("\nSequence lengths:", lengths.tolist())
print("Are lengths sorted descending?",
      bool((lengths[:-1] >= lengths[1:]).all()))

print("\nUnique labels:", labels.unique().tolist())
print("All sequence values finite:", bool(sequences.isfinite().all()))

Sequences shape: torch.Size([16, 3204, 5])
Labels shape: torch.Size([16])
Lengths shape: torch.Size([16])

Sequence data type: torch.float32
Labels data type: torch.float32
Lengths data type: torch.int64

Sequence lengths: [3204, 3204, 3204, 3202, 3200, 3199, 3198, 3197, 3195, 3189, 3186, 3186, 3185, 3184, 3184, 3182]
Are lengths sorted descending? True

Unique labels: [0.0, 1.0]
All sequence values finite: True


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Selected device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Selected device: cuda
GPU: Tesla T4


In [ ]:
from agents.agent2_rnn.src.model import AccountBiGRU

model = AccountBiGRU(
    input_size=CONFIG["input_size"],
    hidden_size=CONFIG["hidden_size"],
    num_layers=CONFIG["num_layers"],
    dropout=CONFIG["dropout"],
).to(device)

print(model)

AccountBiGRU(
  (gru): GRU(5, 64, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=128, out_features=1, bias=True)
)


In [ ]:
total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(f"\nTrainable parameters: {total_params:,}")


Trainable parameters: 27,393


In [ ]:
from agents.agent2_rnn.src.train import train_model, set_seed

# Reproducibility
set_seed(CONFIG["seed"])

# Reset the sampler before training
train_loader.batch_sampler.epoch = 0

# Launch training
history = train_model(
    model=model,
    train_loader=train_loader,
    validation_loader=validation_loader,
    device=device,
)

In [ ]:
import torch

best_checkpoint_path = OUTPUT_DIR / "best_bigru_safe.pt"

print("Checkpoint exists:", best_checkpoint_path.exists())

checkpoint = torch.load(
    best_checkpoint_path,
    map_location=device,
    weights_only=True,
)

# Restore the best model weights
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("\nBest checkpoint successfully restored!")
print("Best epoch:", checkpoint["epoch"])
print("Validation metrics:", checkpoint["validation_metrics"])

Checkpoint exists: True

Best checkpoint successfully restored!
Best epoch: 8
Validation metrics: {'loss': 0.08698535534490343, 'accuracy': 0.9678456591639871, 'precision': 1.0, 'recall': 0.9328859060402684, 'f1': 0.9652777777777778, 'roc_auc': 0.9944071588366891}


In [ ]:
import pandas as pd

history_path = train.CHECKPOINT_DIR / "training_history.csv"

print("History file exists:", history_path.exists())

if history_path.exists():
    history_df = pd.read_csv(history_path)
    display(history_df)

History file exists: True


,epoch,train_loss,val_loss,val_accuracy,val_precision,val_recall,val_f1,val_roc_auc
0,1,0.432579,0.204869,0.932476,0.921053,0.939597,0.930233,0.977877
1,2,0.184524,0.190991,0.932476,0.910256,0.953020,0.931148,0.980529
2,3,0.157569,0.165043,0.945338,0.940000,0.946309,0.943144,0.982351
3,4,0.128123,0.158310,0.951768,0.952703,0.946309,0.949495,0.982973
4,5,0.115827,0.118334,0.964630,0.979167,0.946309,0.962457,0.987489
5,6,0.105001,0.126028,0.964630,0.985915,0.939597,0.962199,0.988524
6,7,0.104317,0.091926,0.964630,0.972603,0.953020,0.962712,0.992336
7,8,0.074628,0.086985,0.967846,1.000000,0.932886,0.965278,0.994407
8,9,0.079180,0.114736,0.958199,1.000000,0.912752,0.954386,0.993330
9,10,0.059512,0.072368,0.980707,1.000000,0.959732,0.979452,0.993620


In [4]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
from pathlib import Path

DATA_DIR = Path(
    "/content/drive/MyDrive/multi-agent-disinformation-detection/data/agent2"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/multi-agent-disinformation-detection/checkpoints/agent2"
)

best_checkpoint_path = OUTPUT_DIR / "best_bigru_safe.pt"
test_data_path = DATA_DIR / "test_scaled_sequences.npz"

print("Data directory:", DATA_DIR)
print("Data directory exists:", DATA_DIR.exists())

print("\nCheckpoint path:", best_checkpoint_path)
print("Checkpoint exists:", best_checkpoint_path.exists())

print("\nTest data path:", test_data_path)
print("Test dataset exists:", test_data_path.exists())

Data directory: /content/drive/MyDrive/multi-agent-disinformation-detection/data/agent2
Data directory exists: True

Checkpoint path: /content/drive/MyDrive/multi-agent-disinformation-detection/checkpoints/agent2/best_bigru_safe.pt
Checkpoint exists: True

Test data path: /content/drive/MyDrive/multi-agent-disinformation-detection/data/agent2/test_scaled_sequences.npz
Test dataset exists: True


In [9]:
import sys
import torch
import numpy as np

sys.path.insert(0, str(PROJECT_DIR))

from agents.agent2_rnn.src import dataset, model, train

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Dataset module:", dataset.__file__)
print("Model module:", model.__file__)
print("Training module:", train.__file__)

PyTorch: 2.14.1+cu130
CUDA available: True
Dataset module: /content/multi-agent-disinformation-detection/agents/agent2_rnn/src/dataset.py
Model module: /content/multi-agent-disinformation-detection/agents/agent2_rnn/src/model.py
Training module: /content/multi-agent-disinformation-detection/agents/agent2_rnn/src/train.py


In [16]:
test_loader = dataset.create_dataloader(
    split="test",
    batch_size=16,
    bucket_size_multiplier=10,
    seed=42
)

print("\nTest DataLoader created successfully!")
print("Number of batches:", len(test_loader))

Loading test dataset...
  Accounts: 312
  Total timesteps: 661,038
  Min length: 15
  Max length: 3,246

Test DataLoader created successfully!
Number of batches: 20


In [17]:
print("test_loader exists:", "test_loader" in globals())

if "test_loader" in globals():
    print("Number of test batches:", len(test_loader))

test_loader exists: True
Number of test batches: 20


In [18]:
sequences, labels, lengths = next(iter(test_loader))

print("Sequences shape:", sequences.shape)
print("Labels shape:", labels.shape)
print("Lengths shape:", lengths.shape)

print("\nData type:", sequences.dtype)
print("Labels:", labels)
print("Lengths:", lengths)

print("\nAll values finite:", torch.isfinite(sequences).all().item())
print("Lengths sorted descending:",
      torch.all(lengths[:-1] >= lengths[1:]).item())

Sequences shape: torch.Size([16, 3246, 5])
Labels shape: torch.Size([16])
Lengths shape: torch.Size([16])

Data type: torch.float32
Labels: tensor([0., 0., 0., 1., 1., 0., 0., 0., 0., 1., 1., 1., 0., 1., 0., 0.])
Lengths: tensor([3246, 3242, 3241, 3240, 3239, 3238, 3238, 3237, 3236, 3236, 3235, 3235,
        3234, 3234, 3233, 3232])

All values finite: True
Lengths sorted descending: True


In [19]:
checkpoint = torch.load(
    best_checkpoint_path,
    map_location="cpu",
    weights_only=True
)

print("Checkpoint epoch:", checkpoint["epoch"])

print("\nSaved model configuration:")
for key, value in checkpoint["config"].items():
    print(f"{key}: {value}")

print("\nSaved validation metrics:")
for key, value in checkpoint["validation_metrics"].items():
    print(f"{key}: {value}")

Checkpoint epoch: 8

Saved model configuration:
input_size: 5
hidden_size: 64
num_layers: 1
dropout: 0.3
batch_size: 16
bucket_size_multiplier: 10
learning_rate: 0.001
weight_decay: 0.01
max_epochs: 30
patience: 5
gradient_clip: 1.0
threshold: 0.5
seed: 42

Saved validation metrics:
loss: 0.08698535534490343
accuracy: 0.9678456591639871
precision: 1.0
recall: 0.9328859060402684
f1: 0.9652777777777778
roc_auc: 0.9944071588366891


In [20]:
# Reconstruct the model using the saved configuration
model_config = checkpoint["config"]

test_model = model.AccountBiGRU(
    input_size=model_config["input_size"],
    hidden_size=model_config["hidden_size"],
    num_layers=model_config["num_layers"],
    dropout=model_config["dropout"]
)

# Load the trained weights
test_model.load_state_dict(checkpoint["model_state_dict"])

# Move model to GPU and switch to evaluation mode
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

test_model = test_model.to(device)
test_model.eval()

print("Model reconstructed successfully!")
print("Device:", device)
print("Architecture:")
print(test_model)

print("\nTraining checkpoint epoch:", checkpoint["epoch"])
print("Model parameters:", sum(p.numel() for p in test_model.parameters()))

Model reconstructed successfully!
Device: cuda
Architecture:
AccountBiGRU(
  (gru): GRU(5, 64, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=128, out_features=1, bias=True)
)

Training checkpoint epoch: 8
Model parameters: 27393


In [23]:
train.CONFIG["threshold"] = checkpoint["config"]["threshold"]

criterion = torch.nn.BCEWithLogitsLoss()

print("Evaluation device:", device)
print("Evaluation threshold:", train.CONFIG["threshold"])
print("Test accounts:", len(test_loader.dataset))
print("Test batches:", len(test_loader))

Evaluation device: cuda
Evaluation threshold: 0.5
Test accounts: 312
Test batches: 20


In [26]:
test_metrics = train.evaluate(
    model=test_model,
    dataloader=test_loader,
    criterion=criterion,
    device=device
)

print("===== AGENT 2 — TEST RESULTS =====")

for metric, value in test_metrics.items():
    print(f"{metric:12s}: {value:.4f}")

===== AGENT 2 — TEST RESULTS =====
loss        : 0.0658
accuracy    : 0.9808
precision   : 1.0000
recall      : 0.9597
f1          : 0.9795
roc_auc     : 0.9964


In [28]:
rf_baseline_model = baseline.run_random_forest_baseline()

print("\nBaseline model recovered successfully:",
      type(rf_baseline_model).__name__)


RANDOM FOREST ACTIVITY-VOLUME BASELINE

Loading training dataset...
Loading train dataset...
  Accounts: 1,451
  Total timesteps: 3,132,339
  Min length: 13
  Max length: 3,250

Loading validation dataset...
Loading validation dataset...
  Accounts: 311
  Total timesteps: 656,018
  Min length: 3
  Max length: 3,249

Training Random Forest...
Training completed.

Training performance:
Accuracy: 0.9235
F1-score: 0.9177
ROC-AUC:  0.9784

Evaluating on validation data...

RANDOM FOREST VALIDATION RESULTS
Accuracy:  0.8553
Precision: 0.8881
Recall:    0.7987
F1-score:  0.8410
ROC-AUC:   0.8930

Confusion Matrix:
[[147  15]
 [ 30 119]]

Classification Report:
              precision    recall  f1-score   support

     Genuine       0.83      0.91      0.87       162
     Spambot       0.89      0.80      0.84       149

    accuracy                           0.86       311
   macro avg       0.86      0.85      0.85       311
weighted avg       0.86      0.86      0.85       311


Baseline 

In [29]:
# Load test account data
test_dataset = dataset.AccountSequenceDataset("test")

# Extract activity-volume feature only
X_test_rf = np.log1p(test_dataset.lengths).reshape(-1, 1)

# Ground-truth labels
y_test_rf = test_dataset.labels

print("Test feature shape:", X_test_rf.shape)
print("Test labels shape:", y_test_rf.shape)

print("\nFeature preview:")
print(X_test_rf[:5])

print("\nClass distribution:")
print("Genuine:", np.sum(y_test_rf == 0))
print("Bots:", np.sum(y_test_rf == 1))

Loading test dataset...
  Accounts: 312
  Total timesteps: 661,038
  Min length: 15
  Max length: 3,246
Test feature shape: (312, 1)
Test labels shape: (312,)

Feature preview:
[[8.06902933]
 [8.07464908]
 [8.07620453]
 [7.73543335]
 [7.79482315]]

Class distribution:
Genuine: 163
Bots: 149


In [30]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Predictions
rf_test_predictions = rf_baseline_model.predict(X_test_rf)
rf_test_probabilities = rf_baseline_model.predict_proba(X_test_rf)[:, 1]

# Evaluation metrics
rf_test_metrics = {
    "accuracy": accuracy_score(y_test_rf, rf_test_predictions),
    "precision": precision_score(y_test_rf, rf_test_predictions, zero_division=0),
    "recall": recall_score(y_test_rf, rf_test_predictions, zero_division=0),
    "f1": f1_score(y_test_rf, rf_test_predictions, zero_division=0),
    "roc_auc": roc_auc_score(y_test_rf, rf_test_probabilities)
}

print("===== RANDOM FOREST — TEST RESULTS =====")

for metric, value in rf_test_metrics.items():
    print(f"{metric:12s}: {value:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_rf, rf_test_predictions))

print("\nClassification Report:")
print(classification_report(
    y_test_rf,
    rf_test_predictions,
    target_names=["Genuine", "Spambot"],
    zero_division=0
))

===== RANDOM FOREST — TEST RESULTS =====
accuracy    : 0.8686
precision   : 0.9154
recall      : 0.7987
f1          : 0.8530
roc_auc     : 0.9047

Confusion Matrix:
[[152  11]
 [ 30 119]]

Classification Report:
              precision    recall  f1-score   support

     Genuine       0.84      0.93      0.88       163
     Spambot       0.92      0.80      0.85       149

    accuracy                           0.87       312
   macro avg       0.88      0.87      0.87       312
weighted avg       0.87      0.87      0.87       312



In [32]:
# Select one test account
sample_index = 0

original_sequence = test_dataset.sequences[sample_index]
original_length = test_dataset.lengths[sample_index]

print("Account index:", sample_index)
print("Sequence shape:", original_sequence.shape)
print("Recorded length:", original_length)
print("Label:", test_dataset.labels[sample_index])

print("\nFirst 5 timesteps:")
print(original_sequence[:5])

print("\nLast 5 timesteps:")
print(original_sequence[-5:])

print("\nData type:", original_sequence.dtype)
print("All values finite:", np.isfinite(original_sequence).all())

Account index: 0
Sequence shape: (3193, 5)
Recorded length: 3193
Label: 1

First 5 timesteps:
[[-2.6192486  -0.5804829  -1.2201297  -0.6276418  -1.2104851 ]
 [-1.1964418  -0.5804829  -1.2201297  -0.6276418  -1.2104851 ]
 [-1.0951976  -0.5804829  -1.2201297  -0.6276418  -1.2104851 ]
 [-0.37890056 -0.5804829  -1.2201297  -0.6276418  -1.2104851 ]
 [-1.3873719  -0.5804829  -1.2201297  -0.6276418  -1.2104851 ]]

Last 5 timesteps:
[[ 0.49974522 -0.86640805 -0.9878357   1.3687643  -0.25092703]
 [-0.35893726 -0.86640805 -0.9878357   1.3687643  -0.25092703]
 [ 0.17052855 -1.0858066  -0.68510365  1.3687643  -0.25092703]
 [-0.12451431 -1.0858066  -0.68510365  1.3687643  -0.25092703]
 [-0.0730153  -1.0858066  -0.68510365  1.3687643  -0.25092703]]

Data type: float32
All values finite: True


In [34]:
from torch.utils.data import DataLoader

# Reproducible random generator
rng = np.random.default_rng(seed=42)

# Create a separate shuffled copy of every account sequence
shuffled_sequences = []

for sequence in test_dataset.sequences:

    # Shuffle complete timestep rows, preserving all 5 features together
    indices = rng.permutation(len(sequence))

    shuffled_sequence = sequence[indices].copy()

    shuffled_sequences.append(shuffled_sequence)


# Pair shuffled sequences with their original labels
shuffled_samples = [
    (
        torch.from_numpy(sequence),
        torch.tensor(label, dtype=torch.float32)
    )
    for sequence, label in zip(
        shuffled_sequences,
        test_dataset.labels
    )
]


# Build the shuffled test DataLoader
shuffled_test_loader = DataLoader(
    shuffled_samples,
    batch_size=16,
    shuffle=False,
    collate_fn=dataset.collate_account_sequences
)

print("Shuffled test dataset created successfully.")
print("Number of accounts:", len(shuffled_samples))
print("Number of batches:", len(shuffled_test_loader))

# Verify that sequence lengths and labels are unchanged
shuffled_lengths = np.array([
    len(sequence) for sequence in shuffled_sequences
])

print("\nOriginal lengths preserved:",
      np.array_equal(test_dataset.lengths, shuffled_lengths))

print("Original labels preserved:",
      np.array_equal(test_dataset.labels,
                     np.array([label.item() for _, label in shuffled_samples])))

print("Original test dataset unchanged:",
      np.array_equal(
          test_dataset.sequences[0],
          original_sequence
      ))

Shuffled test dataset created successfully.
Number of accounts: 312
Number of batches: 20

Original lengths preserved: True
Original labels preserved: True
Original test dataset unchanged: True


In [35]:
shuffled_test_metrics = train.evaluate(
    model=test_model,
    dataloader=shuffled_test_loader,
    criterion=criterion,
    device=device
)

print("===== BIGRU — SHUFFLED TEST RESULTS =====")

for metric, value in shuffled_test_metrics.items():
    print(f"{metric:12s}: {value:.4f}")

===== BIGRU — SHUFFLED TEST RESULTS =====
loss        : 1.3143
accuracy    : 0.5224
precision   : 0.0000
recall      : 0.0000
f1          : 0.0000
roc_auc     : 0.9117


In [36]:
all_shuffled_probabilities = []
all_shuffled_labels = []

test_model.eval()

with torch.no_grad():
    for sequences, labels, lengths in shuffled_test_loader:

        sequences = sequences.to(device)
        lengths = lengths.to(device)

        logits = test_model(sequences, lengths)
        probabilities = torch.sigmoid(logits)

        all_shuffled_probabilities.extend(
            probabilities.cpu().numpy()
        )

        all_shuffled_labels.extend(
            labels.numpy()
        )

all_shuffled_probabilities = np.array(all_shuffled_probabilities)
all_shuffled_labels = np.array(all_shuffled_labels)

shuffled_predictions = (
    all_shuffled_probabilities >= 0.5
).astype(int)

print("===== SHUFFLED PREDICTION DIAGNOSTICS =====")

print("\nPredicted classes:")
print("Genuine:", np.sum(shuffled_predictions == 0))
print("Bots:", np.sum(shuffled_predictions == 1))

print("\nProbability distribution:")
print("Minimum:", np.min(all_shuffled_probabilities))
print("Maximum:", np.max(all_shuffled_probabilities))
print("Mean:", np.mean(all_shuffled_probabilities))
print("Median:", np.median(all_shuffled_probabilities))

print("\nProbability percentiles:")
for percentile in [0, 25, 50, 75, 90, 95, 99, 100]:
    print(
        f"{percentile:3d}%: "
        f"{np.percentile(all_shuffled_probabilities, percentile):.4f}"
    )

===== SHUFFLED PREDICTION DIAGNOSTICS =====

Predicted classes:
Genuine: 312
Bots: 0

Probability distribution:
Minimum: 0.0016419196
Maximum: 0.44518885
Mean: 0.0548176
Median: 0.024708204

Probability percentiles:
  0%: 0.0016
 25%: 0.0085
 50%: 0.0247
 75%: 0.0693
 90%: 0.1455
 95%: 0.2054
 99%: 0.3470
100%: 0.4452


In [37]:
print("===== PROBABILITIES BY TRUE CLASS =====")

for label, class_name in [(0, "Genuine"), (1, "Bots")]:

    class_probs = all_shuffled_probabilities[
        all_shuffled_labels == label
    ]

    print(f"\n{class_name}:")
    print("Accounts:", len(class_probs))
    print("Mean probability:", np.mean(class_probs))
    print("Median probability:", np.median(class_probs))
    print("Minimum:", np.min(class_probs))
    print("Maximum:", np.max(class_probs))

===== PROBABILITIES BY TRUE CLASS =====

Genuine:
Accounts: 163
Mean probability: 0.015271513
Median probability: 0.010173744
Minimum: 0.0016419196
Maximum: 0.11911164

Bots:
Accounts: 149
Mean probability: 0.09807942
Median probability: 0.070166856
Minimum: 0.004628925
Maximum: 0.44518885
